In [1]:
import os
import sys
import re
import random
import threading, time
import ast
import numpy as np
import math
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import subprocess
from multiprocessing import Process
import concurrent.futures
import itertools

from pysat.formula import CNF
from file_gen import*

module_path = os.path.abspath(os.path.join(".."))

In [2]:
################ Generate SATISFIABLE K-SAT instances with arbitrary topology ###############################

#######Enter SAT topology here#########
nvar = 10
nclause = 43
k = 3
instance_id = 1
num_cores = 30
cnf_data_dir = os.path.join(module_path,"data/cnf/")
cnf_file_name = 'uf'+str(nvar)+'-0'+str(instance_id)+'.cnf'
#######################################

#####################################################################################################################
max_iterations = 50
max_restarts = 10
max_flips = 1000
p1 = 0.5
p2 = 0.5
params = {'max_restarts' : max_restarts, 'max_flips' : max_flips, 'p1' : p1, 'p2' : p2,  'heuristics' : "WB"}
#####################################################################################################################

sat_flag = 0
num_iter = 0
while sat_flag!=1 and num_iter<max_iterations:
    num_iter+=1
    clause_mat = gen_kSATprob(nvar,nclause,k)
    avg_flips, prob_s, std_flips, tts_99, walksat_result, _, _, _ = WalkSat_Solver_full_multiCore(clause_mat,params,num_cores)
    if prob_s!=0:
        sat_flag = 1
        clausemat2CNF(clause_mat,cnf_data_dir,cnf_file_name)
    
if sat_flag==1:
    print("Generated a satisfiable SAT instance...")
else:
    print("TIMEOUT. Failed to generate a satisfiable SAT instance...")

Generated a satisfiable SAT instance...


In [3]:
################Generate Single BLIF File based on Specified SAT Instance###############################

cnf_data_dir = os.path.join(module_path,"data/cnf/")
cnf_file_name = 'uf50-01.cnf'

blif_data_dir = os.path.join(module_path,"data/blif/")
blif_filename = 'uf50-01.blif'

formula = CNF(from_file=cnf_data_dir+cnf_file_name)
nvar = formula.nv
clauses = np.array(formula.clauses)

create_blif(nvar,clauses,blif_data_dir+blif_filename)

In [5]:
################Generate Single Activity File based on Specified BLIF File###############################

#Activity files are required for power estimation of VPR routing architecture

blif_data_dir = os.path.join(module_path,"data/blif/")
blif_filename = 'uf50-01.blif'
act_data_dir = os.path.join(module_path,"data/act/")
act_filename = 'uf50-01.act'

nvar = 50
clock_name = "pclk"
nets = extract_blif_nets(blif_data_dir+blif_filename)
write_activity_file(nets,act_data_dir+act_filename,nvar,clock_name=clock_name)

In [7]:
################ Generate BLIF and activity files for all SAT instances in /data/cnf/ ###############################

cnf_data_dir = os.path.join(module_path,"data/cnf/")
blif_data_dir = os.path.join(module_path,"data/blif/")
act_data_dir = os.path.join(module_path,"data/act/")
clock_name = "pclk"

for root, dirs, files in os.walk(cnf_data_dir):
    for file in files:
        formula = CNF(from_file=cnf_data_dir+file)
        nvar = formula.nv
        clauses = np.array(formula.clauses)
        
        blif_filename = file.split(".cnf")[0]+'.blif'
        act_filename = file.split(".cnf")[0]+'.act'
        
        create_blif(nvar,clauses,blif_data_dir+blif_filename)
        
        nets = extract_blif_nets(blif_data_dir+blif_filename)
        write_activity_file(nets,act_data_dir+act_filename,nvar,clock_name=clock_name)